In [1]:
# import packages and specify notebook settings

import requests
import os
import h5py
import random
import pprint
import glob

import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree       # we opt to use cKDTree instead of KDTree for faster performancein C++/Cython

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern']

In [2]:
# specifications
API_KEY = "2b53ab2137136266330440cdef40b53a"
#SIMULATION = "TNG300-1"                                                # simulation volume, select from the following: TNG50-1, TNG100-1, TNG300-1
SIMULATION = "TNG100-1"  
SNAPSHOT = 50                                                          # this snapshot corresponds to redshift z = 1
OUTPUT_DIR = f"{SIMULATION}_groupcat_{SNAPSHOT}"
#OUTPUT_DIR = f"/theory/lts/gbelinar/IllustrisTNG/{SIMULATION}_groupcat_{SNAPSHOT}"

HEADERS = {"api-key": API_KEY}

os.makedirs(OUTPUT_DIR, exist_ok=True)                                  # directory for groupcatalog files


# first, get snapshot metadata to guide query
snapshot_url = f"https://www.tng-project.org/api/{SIMULATION}/snapshots/{SNAPSHOT}/"

response = requests.get(snapshot_url, headers=HEADERS)
response.raise_for_status()
metadata = response.json()                                              # extract metadata of the chosen snapshot

print("Redshift:", metadata["redshift"])                                # verify redshift of the snapshot


# get group catalog
groupcat_url = metadata["files"]["groupcat"]
print("Group Catalog URL:", groupcat_url)


# obtain list of group catalog files
response = requests.get(groupcat_url, headers=HEADERS)
response.raise_for_status()
groupcat_info = response.json()

files = groupcat_info["files"]

print(f"Number of Group Catalog files: {len(files)}")


Redshift: 0.99729422578194
Group Catalog URL: http://www.tng-project.org/api/TNG100-1/files/groupcat-50/
Number of Group Catalog files: 448


In [7]:
files[:7]

['http://www.tng-project.org/api/TNG100-1/files/groupcat-50.0.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.1.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.2.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.3.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.4.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.5.hdf5',
 'http://www.tng-project.org/api/TNG100-1/files/groupcat-50.6.hdf5']

In [ ]:
#```python
# download all HDF5 chunks

for url in files[:7]:

    # Extract filename from URL
    filename = url.split("/")[-1]

    filepath = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(filepath):
        print(f"The file {filename} already exists. Proceeding to next file ...")
        continue

    print(f"Downloading {filename} ...")

    r = requests.get(url, headers=HEADERS, stream=True)
    r.raise_for_status()

    with open(filepath, "wb") as f:

        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
            
    print(f"Downloading {filename} complete.")

print(f"Downloading {SIMULATION}_groupcat_{SNAPSHOT} complete.")
#```


The file groupcat-50.0.hdf5 already exists. Proceeding to next file ...
The file groupcat-50.1.hdf5 already exists. Proceeding to next file ...
The file groupcat-50.2.hdf5 already exists. Proceeding to next file ...
The file groupcat-50.3.hdf5 already exists. Proceeding to next file ...
The file groupcat-50.4.hdf5 already exists. Proceeding to next file ...
The file groupcat-50.5.hdf5 already exists. Proceeding to next file ...


In [ ]:
files = sorted(glob.glob(OUTPUT_DIR + "/*.hdf5"))
#files

In [ ]:
group_pos = []
group_mass = []
group_r200 = []

subhalo_pos = []
subhalo_sfr = []

In [ ]:
d = h5py.File(files[0], "r")

In [ ]:
d['Group'].keys()

In [ ]:
for filename in files:

    with h5py.File(filename, "r") as f:

        group_pos.append(f["Group"]["GroupPos"][:])
        group_mass.append(f["Group"]["Group_M_Crit200"][:])
        group_r200.append(f["Group"]["Group_R_Crit200"][:])

        subhalo_pos.append(f["Subhalo"]["SubhaloPos"][:])
        subhalo_sfr.append(f["Subhalo"]["SubhaloSFR"][:])

        # Stellar Mass, Subhalo Mass

In [ ]:
# check how data is initially structured using one element

print(group_pos[0], "\n" ,group_pos[0].shape, "\n")
print(group_mass[0], "\n" ,group_mass[0].shape, "\n")
print(group_r200[0], "\n" ,group_r200[0].shape, "\n")

print(subhalo_pos[0], "\n" ,subhalo_pos[0].shape, "\n")
print(subhalo_sfr[0], "\n" ,subhalo_sfr[0].shape, "\n")

In [ ]:
group_pos = np.concatenate(group_pos, axis=0)
group_mass = np.concatenate(group_mass, axis=0)
group_r200 = np.concatenate(group_r200, axis=0)

subhalo_pos = np.concatenate(subhalo_pos, axis=0)
subhalo_sfr = np.concatenate(subhalo_sfr, axis=0)

In [ ]:
# inspect file structure, in particular, the keys related to physical attribute
with h5py.File(files[0], 'r') as f:
    print(f.keys(), "\n")

with h5py.File(files[0], 'r') as f:
    print(list(f["Group"].keys()), "\n")

with h5py.File(files[0], 'r') as f:
    print(list(f["Subhalo"].keys()))

In [ ]:
# verify structure of the lists, everything should be "unwrapped"

print(group_pos.shape)
print(group_mass.shape)
print(group_r200.shape)

print(subhalo_pos.shape)
print(subhalo_sfr.shape)

In [ ]:
# convert group_mass into solar mass units

h = 0.6774

M200 = group_mass * (1e10/h)

print(np.min(M200), np.max(M200))


In [ ]:
# filter clusters with masses greater than 10^14 solar masses

cluster_mask = M200 > 1e14 # set condition

cluster_ids = np.where(cluster_mask)[0]

print(len(M200), len(cluster_ids))

In [ ]:
# check out one cluster

halo = cluster_ids[0]

print(group_pos[halo], M200[halo], group_r200[halo])

In [ ]:
# build KDTree
tree = cKDTree(subhalo_pos)

In [ ]:
# query one cluster
center = group_pos[halo]
radius = group_r200[halo]

members = tree.query_ball_point(center, radius)

print(len(members))

In [ ]:
# checked the filtered galaxy
member_sfr = subhalo_sfr[members]

member_sfr[:1000]

In [ ]:
# try to visualize some result

member_pos = subhalo_pos[members]

plt.figure(figsize=(6,6))

plt.scatter(member_pos[:,0], member_pos[:,1], s=5)

plt.scatter(center[0], center[1], c="red", s=100)

plt.xlabel("$x$")
plt.ylabel("$y$")

plt.show()